In [ ]:
import os

from cuda import core
import numpy as np
import tensorrt as trt
from cuda.bindings import runtime as cudart

from typing import Tuple, List, Union

import pandas as pd

import monai
from monai.data import itk_torch_bridge as mitb
import monai.transforms as mt
from monai.data import MetaTensor

# from numba import cuda
# import torch

In [ ]:
def trt_dtype_to_np(dtype: trt.DataType):
    return np.dtype(trt.nptype(dtype))

def get_io_names(engine):
    return [engine.get_tensor_name(i) for i in range(engine.num_io_tensors)]

def get_io_info(engine):
    """
    Returns list of input, output dicts
    """
    inputs = []
    outputs = []
    for name in get_io_names(engine):
        info = {
            "name": name,
            "mode": engine.get_tensor_mode(name),
            "dtype": engine.get_tensor_dtype(name),
            "shape": engine.get_tensor_shape(name),
        }
        if(info["mode"] == trt.TensorIOMode.INPUT):
            inputs.append(info)
        else:
            outputs.append(info)
        
    return inputs, outputs

def to_device(x: Union[np.ndarray, MetaTensor], stream: core.Stream):
    if isinstance(x, MetaTensor):
        x_ptr = x.data_ptr()
        x_nbytes = x.nbytes
    else:
        x_ptr = x.ctypes.data
        x_nbytes = x.nbytes

    err, ptr_c = cudart.cudaMalloc(x_nbytes)
    if err != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f"cudaMalloc failed with error code {err}")
    err, = cudart.cudaMemcpyAsync(ptr_c, x_ptr, x_nbytes, cudart.cudaMemcpyKind.cudaMemcpyHostToDevice, stream.handle)
    if err != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f"cudaMemcpyAsync failed with error code {err}")
    return ptr_c

def read_image(fpath: str, transform=mt.Identity()):
    reader = monai.data.ITKReader()    
    itk_img = reader.read(fpath)
    img_t = mitb.itk_image_to_metatensor(itk_img)
    img_t = transform(img_t)
    return img_t

def get_input_transform():
    transform = mt.Compose(
        [   
            mt.Resize(                
                spatial_size=(512, 512),
                mode="bilinear",
                align_corners=False,
            ),
            mt.ScaleIntensityRange(                
                a_min=0,
                a_max=255,
                b_min=0.0,
                b_max=1.0,
                clip=True,
            ),
        ]
    )
    return transform

def get_output_transform(spatial_size: tuple):
    transform = mt.Compose(
        [   
            mt.Resize(                
                spatial_size=spatial_size,
                mode="bilinear",
                align_corners=False,
            ),            
        ]
    )
    return transform

In [ ]:
device = core.Device(0)
device.set_current()
stream = device.create_stream()

In [ ]:
# arr_d = cuda.to_device(arr, stream=stream)
# arr_d.device_ctypes_pointer.value

In [ ]:
# arr_t = torch.from_numpy(arr).cuda()
# arr_t.data_ptr()

In [ ]:
TRT_LOGGER = trt.Logger(trt.Logger.VERBOSE)
engine_file_path = '/MEDUSA_STOR/jprieto/EGower/train_output/segmentation/ttunet/full_seg/v5.0/epoch=153-val_loss=0.14.trt'

trt_runtime = trt.Runtime(TRT_LOGGER)

with open(engine_file_path, 'rb') as f:
    engine = trt_runtime.deserialize_cuda_engine(f.read())

In [ ]:


def infer(x: Union[np.ndarray, MetaTensor], stream: core.Stream, engine: trt.ICudaEngine):    
    
    inputs, outputs = get_io_info(engine)

    assert len(x.shape) == 4  
    assert x.shape[1:] == inputs[0]["shape"][1:], f"Expected input shape {inputs[0]['shape']}, got {x.shape}"

    B, C, H, W = x.shape

    with engine.create_execution_context() as context:

        context.set_input_shape(inputs[0]["name"], x.shape)

        d_input = to_device(x, stream)
        
        # Allocate output buffer
        output_info = outputs[0]
        output_shape = [B] + list(output_info["shape"][1:])

        y_host = np.empty(output_shape, dtype=trt_dtype_to_np(output_info["dtype"]))

        err, d_output = cudart.cudaMalloc(y_host.nbytes)
            
        if err != cudart.cudaError_t.cudaSuccess:
            raise RuntimeError(f"cudaMalloc failed with error code {err}")
        
        context.set_tensor_address(inputs[0]["name"], d_input)
        context.set_tensor_address(outputs[0]["name"], d_output)

        context.execute_async_v3(stream.handle)
    
    # Copy output back to host    
    err, = cudart.cudaMemcpyAsync(y_host.ctypes.data, d_output, y_host.nbytes, cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost, stream.handle)
    if err != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f"cudaMemcpyAsync failed with error code {err}")
    
    # Synchronize stream
    stream.sync()
    
    # Free device memory
    cudart.cudaFree(d_input)
    cudart.cudaFree(d_output)

    if isinstance(x, MetaTensor):
        y_host = MetaTensor(y_host)
    
    return y_host

In [ ]:
x = np.random.rand(256, 3, 512, 512).astype(np.float32)

In [ ]:

y = infer(x, stream, engine)
print("Inference completed. Output shape:", y.shape)

In [ ]:
get_io_names(engine)

In [ ]:
engine.get_tensor_shape('input')

In [ ]:
monai.data.image_reader

In [ ]:
mount_point = "/MEDUSA_STOR/jprieto/EGower/"
df = pd.read_csv(os.path.join(mount_point, "CSV_files/mtss_pret_combined_train.csv"))

In [ ]:
fname = os.path.join(mount_point, df.iloc[0]['filename'])
x_t = read_image(fname, transform=get_input_transform())


In [ ]:
x_hat = infer(x_t.unsqueeze(0), stream, engine)

In [ ]:
x_hat.shape

In [ ]:

ot = get_output_transform(spatial_size=x_t.meta["spatial_shape"])
x_out = ot(x_hat.argmax(dim=1, keepdim=True).squeeze(0))

In [ ]:
x_t.shape

In [ ]:
import matplotlib.pyplot as plt

# plot image and overlay

plt.imshow(x_t.permute(1, 2, 0))

In [ ]:
plt.imshow(x_out.squeeze(), cmap='jet')